# Pandas Titanic Cookbook

A self-contained cookbook based on the attached pandas cheat sheet and the supplied Titanic CSV.

It can be run as a script or imported as a module. Set `CSV_PATH` to the location of the supplied `titanic.csv` if it is not in the same directory.

The notebook reads the supplied `titanic.csv` from the current directory.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = Path("titanic.csv")
OUTPUT_DIR = Path("titanic_cookbook_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 3)


def section(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def load_titanic(path=CSV_PATH):
    """Load and lightly standardize the supplied Titanic CSV."""
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    df["Name"] = df["Name"].str.strip()
    df["Ticket"] = df["Ticket"].astype("string").str.strip()
    df["Cabin"] = df["Cabin"].astype("string").str.strip()
    df["Embarked"] = df["Embarked"].astype("string").str.strip()
    return df


def build_features(df):
    """Create reusable analysis variables without modifying the input frame."""
    return (df.assign(
        FamilySize=lambda d: d["SibSp"] + d["Parch"] + 1,
        IsAlone=lambda d: (d["SibSp"] + d["Parch"] == 0).astype("int8"),
        HasCabin=lambda d: d["Cabin"].notna().astype("int8"),
        Title=lambda d: d["Name"].str.extract(r",\s*([^.]*)\.", expand=False).str.strip(),
        FarePerPerson=lambda d: d["Fare"] / (d["SibSp"] + d["Parch"] + 1),
        AgeBand=lambda d: pd.cut(d["Age"], bins=[-np.inf, 12, 18, 35, 60, np.inf],
                                 labels=["child", "adolescent", "adult", "middle_age", "senior"]),
    ))


def run_cookbook(df):
    section("1. Inspect the DataFrame")
    print(df.head())
    print("Shape:", df.shape)
    print("\nDtypes:")
    print(df.dtypes)
    print("\nMissing values:")
    print(df.isna().sum().sort_values(ascending=False))
    print("\nNumeric summary:")
    print(df.describe(numeric_only=True))

    section("2. Subsetting rows and columns")
    print("Female passengers:")
    print(df.loc[df["Sex"].eq("female"), ["PassengerId", "Name", "Pclass", "Age", "Survived"]].head())
    print("\nFirst and last rows:")
    print(df.head(3))
    print(df.tail(3))
    print("\nRows 10-14 by position:")
    print(df.iloc[10:15])
    print("\nSelected columns using a regex:")
    print(df.filter(regex=r"^(Passenger|Surv|Fare|Age)" ).head())
    print("\nSingle values:")
    print(".iat[0, 3] =", df.iat[0, 3])
    print(".at[1, 'Name'] =", df.at[1, "Name"])

    section("3. Boolean logic and query")
    print(df.query("Sex == 'female' and Pclass <= 2")[["Name", "Sex", "Pclass", "Survived"]].head())
    ports = ["C", "Q"]
    print("\nPassengers embarked at Cherbourg or Queenstown:")
    print(df[df["Embarked"].isin(ports)][["Name", "Embarked"]].head())
    print("\nPassengers with missing age:")
    print(df[pd.isnull(df["Age"])][["Name", "Age"]].head())

    section("4. Sorting, duplicates, renaming, and indexing")
    print(df.sort_values("Fare", ascending=False)[["Name", "Pclass", "Fare"]].head())
    renamed = df.rename(columns={"Pclass": "PassengerClass"})
    print("\nRenamed columns:", renamed.columns.tolist())
    indexed = df.set_index("PassengerId").sort_index()
    print("\nSorted index:")
    print(indexed.head(2))
    print("\nReset index:")
    print(indexed.reset_index().head(2))
    print("\nDuplicate rows after appending one copy:", pd.concat([df, df.iloc[[0]]]).duplicated().sum())

    section("5. Tidy data, melt, pivot, and method chaining")
    summary_long = (df.groupby("Pclass", as_index=False)[["Survived", "Fare"]].mean()
                    .melt(id_vars="Pclass", var_name="variable", value_name="value")
                    .rename(columns={"variable": "metric", "value": "mean_value"})
                    .query("mean_value >= 0"))
    print(summary_long)
    survival_wide = (df.groupby(["Pclass", "Sex"], as_index=False)["Survived"].mean()
                     .pivot(index="Pclass", columns="Sex", values="Survived"))
    print("\nSurvival rate in wide format:")
    print(survival_wide)

    section("6. Concatenation and merge")
    first = df.iloc[:3, :5]
    second = df.iloc[3:6, :5]
    print("Rows concatenated:")
    print(pd.concat([first, second], ignore_index=True))
    left = df[["PassengerId", "Name"]].head(5)
    right = df[["PassengerId", "Survived"]].head(5)
    print("\nColumns concatenated:")
    print(pd.concat([left.set_index("PassengerId"), right.set_index("PassengerId")], axis=1))
    ticket_summary = df.groupby("Ticket", as_index=False).agg(
        TicketPassengers=("PassengerId", "size"), TicketMeanFare=("Fare", "mean")
    )
    merged = pd.merge(df, ticket_summary, how="left", on="Ticket")
    print("\nMerged ticket summary:")
    print(merged[["Name", "Ticket", "TicketPassengers", "TicketMeanFare"]].head())

    section("7. Grouping and summarizing")
    grouped = (df.groupby(["Pclass", "Sex"])
               .agg(Passengers=("PassengerId", "size"),
                    SurvivalRate=("Survived", "mean"),
                    MeanAge=("Age", "mean"),
                    MedianFare=("Fare", "median"))
               .reset_index())
    print(grouped)
    print("\nEmbarkation counts:")
    print(df["Embarked"].value_counts(dropna=False))
    print("\nTop and bottom fares:")
    print(df.nlargest(5, "Fare")[["Name", "Fare"]])
    print(df.nsmallest(5, "Fare")[["Name", "Fare"]])

    section("8. Missing data and new columns")
    filled = df.assign(
        AgeFilled=df["Age"].fillna(df["Age"].median()),
        EmbarkedFilled=df["Embarked"].fillna(df["Embarked"].mode().iat[0]),
        CabinKnown=df["Cabin"].notna(),
    )
    print(filled[["Name", "Age", "AgeFilled", "Embarked", "EmbarkedFilled", "CabinKnown"]].head())
    featured = build_features(df)
    print("\nFeature-engineered data:")
    print(featured[["Name", "FamilySize", "IsAlone", "HasCabin", "Title", "FarePerPerson", "AgeBand"]].head())

    section("9. Vectorized operations, mapping, ranking, and windows")
    result = featured.sort_values("PassengerId").copy()
    result["FareRank"] = result["Fare"].rank(method="dense", ascending=False)
    result["FareShifted"] = result["Fare"].shift(1)
    result["FareCumulative"] = result["Fare"].fillna(0).cumsum()
    result["FareRollingMean"] = result["Fare"].rolling(5, min_periods=1).mean()
    result["SexCode"] = result["Sex"].map({"female": 0, "male": 1})
    result["FareClipped"] = result["Fare"].clip(lower=0, upper=100)
    print(result[["PassengerId", "Fare", "FareRank", "FareShifted", "FareCumulative", "FareRollingMean", "SexCode", "FareClipped"]].head(10))

    section("10. String operations and regular expressions")
    text = df[["Name", "Ticket"]].copy()
    text["Surname"] = text["Name"].str.partition(",")[0].str.title()
    text["Title"] = text["Name"].str.extract(r",\s*([^.]*)\.", expand=False).str.strip()
    text["NameLength"] = text["Name"].str.len()
    text["HasPeriodInTicket"] = text["Ticket"].str.contains(r"\.", regex=True, na=False)
    text["TicketPrefix"] = text["Ticket"].str.slice(0, 5)
    print(text.head())

    section("11. Datetime and type conversion")
    dated = df.assign(
        SyntheticBookingDate=pd.date_range("1912-04-01", periods=len(df), freq="D")
    )
    dated["BookingYear"] = dated["SyntheticBookingDate"].dt.year
    dated["BookingMonth"] = dated["SyntheticBookingDate"].dt.month
    dated["BookingDay"] = dated["SyntheticBookingDate"].dt.day
    print(dated[["PassengerId", "SyntheticBookingDate", "BookingYear", "BookingMonth", "BookingDay"]].head())
    print("\nConverted dtypes:")
    print(dated.convert_dtypes().dtypes)

    section("12. Plots")
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    df.groupby("Pclass")["Survived"].mean().plot.bar(ax=axes[0], title="Survival by class", ylim=(0, 1), rot=0)
    df["Fare"].plot.hist(ax=axes[1], bins=30, title="Fare distribution")
    df.plot.scatter(x="Age", y="Fare", c="Survived", cmap="coolwarm", alpha=0.55, ax=axes[2], title="Age versus fare")
    plt.tight_layout()
    plot_path = OUTPUT_DIR / "titanic_overview.png"
    fig.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved plot to", plot_path)

    section("13. Input/output")
    output_csv = OUTPUT_DIR / "titanic_featured.csv"
    featured.to_csv(output_csv, index=False)
    reloaded = pd.read_csv(output_csv)
    print("Saved and reloaded:", output_csv)
    print(reloaded.shape)
    try:
        output_parquet = OUTPUT_DIR / "titanic_featured.parquet"
        featured.to_parquet(output_parquet, index=False)
        print("Saved:", output_parquet)
    except ImportError:
        print("Parquet skipped: install pyarrow or fastparquet to enable it.")

    section("14. Practice exercises")
    print("1. Which embarkation port has the highest survival rate?")
    print("2. Compare survival by Pclass and Sex using a pivot table.")
    print("3. Extract titles from Name and count them.")
    print("4. Find passengers travelling alone who paid above the median fare.")
    print("5. Calculate the median fare per ticket and merge it back to the passenger rows.")
    print("6. Create three Fare quantiles with pd.qcut and compare survival across them.")
    print("7. Use a left-only merge with indicator=True to identify unmatched ticket metadata.")
    print("8. Save a compact table containing PassengerId, Survived, Pclass, Sex, AgeBand, and FarePerPerson.")

    return featured, grouped, survival_wide


if __name__ == "__main__":
    titanic = load_titanic()
    featured, grouped, survival_wide = run_cookbook(titanic)
